# Evaluation with RAGAS

#### RAG Chain

In [ ]:
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore, RetrievalMode, FastEmbedSparse
from langchain_community.embeddings import InfinityEmbeddings
from langchain_ollama import ChatOllama

embeddings = InfinityEmbeddings(
    model="AITeamVN/Vietnamese_Embedding",
    infinity_api_url="http://192.168.88.179:2025",
)
collection_name = "lessons_learned"
vectorstore = QdrantVectorStore(
    client=QdrantClient(url="http://192.168.88.179:6333"),
    collection_name=collection_name,
    embedding=embeddings,
    vector_name="dense",
    sparse_embedding=FastEmbedSparse(),
    sparse_vector_name="sparse",
    retrieval_mode=RetrievalMode.HYBRID,
)

llm = ChatOllama(
    model="qwen3.5:9b",
    base_url="http://192.168.88.179:11435",
    keep_alive=-1,
    seed=9999,
    num_ctx=32768,
    reasoning=False,
    temperature=0,
)

In [ ]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
relevant_docs = base_retriever.invoke("What is Retrieval Augmented Generation?")
len(relevant_docs)

In [ ]:
from langchain_core.documents import Document

points = vectorstore.client.query_points(collection_name, limit=1000).points
docs = [
    Document(
        point.payload.get("page_content", ""),
        metadata=point.payload.get("metadata") or {},
    )
    for point in points
]
print(len(docs))

In [ ]:
from operator import itemgetter
from langchain_classic.schema.runnable import RunnablePassthrough
from langchain_classic.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""\
Answer the question based only on the following context. If you cannot answer the question with the context, please respond with 'I don't know':

### CONTEXT
{context}

### QUESTION
{question}
""")

rag_pipeline = (
    {
        "context": itemgetter("question") | base_retriever,
        "question": itemgetter("question"),
    }
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": prompt | llm | StrOutputParser(), "context": itemgetter("context")}
)

In [ ]:
question = "Sự cố liên quan đến Amazon SNS là gì?"

result = rag_pipeline.invoke({"question" : question})

print(result)

#### RAG API

In [ ]:
from dataclasses import dataclass, field
from langgraph_sdk import get_sync_client
from langgraph_sdk.client import SyncLangGraphClient
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage
from langchain_community.embeddings import InfinityEmbeddings
from langchain_ollama import ChatOllama
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore, RetrievalMode, FastEmbedSparse


@dataclass
class RAGAgent:
    _client: SyncLangGraphClient = field(
        default_factory=lambda: get_sync_client(
            url="http://localhost:2024", timeout=120
        ),
        repr=False,
    )

    def __post_init__(self):
        pass

    def invoke(self, input: dict) -> dict:
        prompt = input.get("question", "")
        resp = self._client.runs.wait(
            thread_id=None,
            assistant_id="chat",
            input={"messages": [HumanMessage(prompt)]},
        )
        messages = resp.get("messages") or []
        docs: list[dict] = resp.get("context") or []
        print(len(docs))

        try:
            response = AIMessage.model_validate(messages[-1])
            context = [Document(doc.get("page_content"), metadata=doc.get("metadata")) for doc in docs]
        except Exception as e:
            print(f"Error parsing AIMessage: {e}")
            response = AIMessage(content="")
            context = []

        return {
            "response": response.content,
            "context": context,
        }


rag_pipeline = RAGAgent()

embeddings = InfinityEmbeddings(
    model="AITeamVN/Vietnamese_Embedding",
    infinity_api_url="http://192.168.88.179:2025",
)

llm = ChatOllama(
    model="qwen3.5:9b",
    base_url="http://192.168.88.179:11435",
    keep_alive=-1,
    seed=9999,
    num_ctx=32768,
    reasoning=False,
    temperature=0,
)
collection_name = "lessons_learned"
vectorstore = QdrantVectorStore(
    client=QdrantClient(url="http://192.168.88.179:6333"),
    collection_name=collection_name,
    embedding=embeddings,
    vector_name="dense",
    sparse_embedding=FastEmbedSparse(),
    sparse_vector_name="sparse",
    retrieval_mode=RetrievalMode.HYBRID,
)

In [ ]:
question = "Sự cố liên quan đến Amazon SNS là gì?"

result = rag_pipeline.invoke({"question" : question})

print(result)

In [ ]:
from langchain_core.documents import Document

points = vectorstore.client.query_points(collection_name, limit=1000).points
docs = [
    Document(
        point.payload.get("page_content", ""),
        metadata=point.payload.get("metadata") or {},
    )
    for point in points
]
print(len(docs))

### Build Testset

In [ ]:
import typing as t
from enum import Enum
from ragas.prompt import PydanticPrompt
from ragas.testset.graph import Node
from ragas.testset.persona import Persona, PersonaList
from ragas.testset.synthesizers.single_hop.prompts import (
    QueryCondition,
    GeneratedQueryAnswer,
)
from ragas.testset.synthesizers.single_hop.specific import (
    SingleHopSpecificQuerySynthesizer,
)

class QueryAnswerGenerationPrompt(PydanticPrompt[QueryCondition, GeneratedQueryAnswer]):
    instruction: str = (
        "Tạo một truy vấn và câu trả lời một bước dựa trên các điều kiện đã chỉ định (nhân vật, thuật ngữ, phong cách, độ dài) và ngữ cảnh được cung cấp. "
        "Đảm bảo câu trả lời hoàn toàn phù hợp với ngữ cảnh, chỉ sử dụng thông tin trực tiếp từ ngữ cảnh được cung cấp.\n"
        "### Hướng dẫn:\n"
        "1. **Tạo Truy vấn**: Dựa trên ngữ cảnh, nhân vật, thuật ngữ, phong cách và độ dài, hãy tạo một câu hỏi phù hợp với quan điểm của nhân vật và kết hợp thuật ngữ.\n"
        "2. **Tạo Câu trả lời**: Chỉ sử dụng nội dung từ ngữ cảnh được cung cấp, hãy xây dựng một câu trả lời chi tiết cho truy vấn. "
        "Không thêm bất kỳ thông tin nào không có trong hoặc không thể suy ra từ ngữ cảnh.\n"
    )
    input_model: t.Type[QueryCondition] = QueryCondition
    output_model: t.Type[GeneratedQueryAnswer] = GeneratedQueryAnswer
    examples: t.List[t.Tuple[QueryCondition, GeneratedQueryAnswer]] = [
        (
            QueryCondition(
                persona=Persona(
                    name="Software Engineer",
                    role_description="Tập trung vào các thực hành tốt nhất về lập trình và thiết kế hệ thống.",
                ),
                term="Bài học kinh nghiệm",
                query_style="Câu hỏi sự cố",
                query_length="Vừa",
                context=(
                    "Khi triển khai kiểm tra hoạt động của module CARECONNE có thay đổi xử lý dùng chung: bước kiểm tra hoạt động của datacenter. "
                    "Hiệu chỉnh này đã thiếu sót xử lý ở kết quả trả về ở module HN, gây vấn đề Alive Monitoring gửi email thông báo lỗi ở HN mặc dù thực tế không có lỗi xảy ra."
                ),
            ),
            GeneratedQueryAnswer(
                query="Nguyên nhân là gì dẫn đến việc Alive Monitoring gửi email thông báo lỗi đến HN?",
                answer="Nguyên nhân là do hiệu chỉnh trong module CARECONNE đã thiếu sót xử lý ở kết quả trả về ở module HN, dẫn đến việc Alive Monitoring gửi email thông báo lỗi mặc dù thực tế không có lỗi xảy ra.",
            ),
        ),
    ]


class QueryLength(str, Enum):
    """
    Enumeration of query lengths. Available options are: MEDIUM, SHORT
    """

    MEDIUM = "medium"
    SHORT = "short"


class QueryStyle(str, Enum):
    """
    Enumeration of query styles. Available options are: PERFECT_GRAMMAR, POOR_GRAMMAR, WEB_SEARCH_LIKE
    """

    PERFECT_GRAMMAR = "Perfect grammar"
    POOR_GRAMMAR = "Poor grammar"
    WEB_SEARCH_LIKE = "Web search like queries"


class SingleHopIncidentQuerySynthesizer(SingleHopSpecificQuerySynthesizer):
    def prepare_combinations(
        self,
        node: Node,
        terms: t.List[str],
        personas: t.List[Persona],
        persona_concepts: t.Dict[str, t.List[str]],
    ) -> t.List[t.Dict[str, t.Any]]:

        sample = {"terms": terms, "node": node}
        valid_personas = []
        persona_list = PersonaList(personas=personas)
        for persona, concepts in persona_concepts.items():
            concepts = [concept.lower() for concept in concepts]
            if any(term.lower() in concepts for term in terms):
                if persona_list[persona]:
                    valid_personas.append(persona_list[persona])
        sample["personas"] = valid_personas
        sample["styles"] = list(QueryStyle)
        sample["lengths"] = list(QueryLength)

        return [sample]

In [ ]:
from ragas.testset.transforms import default_transforms, apply_transforms
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

run_config = RunConfig(timeout=3600, max_retries=1, seed=9999)
transformer_llm = LangchainLLMWrapper(llm, run_config)
embedding_model = LangchainEmbeddingsWrapper(embeddings, run_config)

In [ ]:
from ragas.testset.synthesizers import default_query_distribution

query_distribution = [
    (
        SingleHopIncidentQuerySynthesizer(
            llm=transformer_llm,
            generate_query_reference_prompt=QueryAnswerGenerationPrompt(),
        ),
        1.0,
    )
]

query_distribution

In [ ]:
from ragas.testset.graph import KnowledgeGraph
from ragas.testset.graph import Node, NodeType

kg = KnowledgeGraph()
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata},
        )
    )

trans = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, trans, run_config)

kg.save("knowledge_graph.json")

In [ ]:
from ragas.testset.graph import KnowledgeGraph
from ragas.testset import TestsetGenerator

loaded_kg = KnowledgeGraph.load("knowledge_graph.json")
generator = TestsetGenerator(
    llm=transformer_llm,
    embedding_model=embedding_model,
    knowledge_graph=loaded_kg,
    persona_list=[
        Persona(
            name="Software Engineer",
            role_description="Tập trung vào các thực hành tốt nhất về lập trình và thiết kế hệ thống.",
        ),
    ],
)
testset = generator.generate(testset_size=100, query_distribution=query_distribution)

In [ ]:
ragas_testset = testset.to_evaluation_dataset()
print("Query:", ragas_testset[0].user_input)
print("Reference:", ragas_testset[0].reference)
ragas_testset.to_jsonl("ragas_testset.jsonl")

### Evaluate

In [ ]:
from tqdm import tqdm
from ragas import evaluate, EvaluationDataset
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
)
from langchain_classic.schema.runnable import RunnableSerializable


def format_content(doc: Document) -> str:
    page_content = doc.page_content.strip()

    if not page_content:
        return "Không tìm thấy tài liệu phù hợp."

    source = doc.metadata["source"]
    page = doc.metadata["page_number"]
    project = doc.metadata["project_name"]
    occurred_at = doc.metadata["occurred_at"]

    output = (
        f"[SOURCE] {source}#page={page} - project={project.capitalize()} - occurred_at={occurred_at}\n"
        f"{page_content}"
    )

    return output

def evaluate_ragas(
    rag_pipeline: RunnableSerializable | RAGAgent, testset: EvaluationDataset
):
    for row in tqdm(testset):
        answer = rag_pipeline.invoke({"question": row.user_input})
        response: str = answer.get("response") or ""
        context: list[Document] = answer.get("context") or []
        row.response = response
        row.retrieved_contexts = [format_content(doc) for doc in context]

    print(f"Question: {testset[0].user_input}")
    print(f"Response: {testset[0].response}")
    print(f"Reference: {testset[0].reference}")
    print(f"Retrieved Contexts: {testset[0].retrieved_contexts}")
    print(f"Reference Contexts: {testset[0].reference_contexts}")

    result = evaluate(
        testset,
        metrics=[
            Faithfulness(),
            AnswerRelevancy(),
            ContextPrecision(),
            ContextRecall(),
        ],
        llm=transformer_llm,
        embeddings=embedding_model,
        run_config=run_config,
        raise_exceptions=True,
    )
    return result

In [ ]:
from ragas import EvaluationDataset

ragas_testset = EvaluationDataset.from_jsonl("ragas_testset.jsonl")
result = evaluate_ragas(rag_pipeline, ragas_testset)
result